# MachSense - Exploratory Data Analysis (EDA)
## Predictive Maintenance & Sensor Health Diagnostics

**Objective**: Conduct an in-depth exploratory analysis of the manufacturing sensor telemetry dataset to uncover underlying physical patterns, quantify class imbalance, detect anomalies and multi-collinearities, and formulate evidence-based feature engineering hypotheses for predictive maintenance.

### 1. Environment Setup & Data Ingestion
We load project configurations dynamically without hardcoded paths, ensuring clean reproducibility.

In [ ]:
import sys
from pathlib import Path

# Ensure project root is in sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from machsense.config.settings import get_settings
from machsense.data.loader import load_raw_data
from machsense.data.eda import compute_dataset_summary, compute_domain_features, compute_correlations

# Matplotlib configuration
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 10

# Load clean dataset
df = load_raw_data()
print(f"Loaded dataset with {df.shape[0]:,} rows and {df.shape[1]} columns.")
df.head()

### 2. Dataset Overview & Data Integrity Audit
We inspect column types, missingness, and verify physical integrity.

In [ ]:
summary = compute_dataset_summary(df)
print(summary.summary())

print("\n--- Data Types & Non-Null Values ---")
df.info()

### 3. Target Distribution & Class Imbalance Analysis
Machine failure occurs in only **~3.39%** of records. This severe imbalance has major consequences for metric selection.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Target distribution count
target_counts = df["machine_failure"].value_counts()
colors = ["#2b5c8f", "#d9534f"]
axes[0].bar(["Normal (0)", "Failure (1)"], target_counts.values, color=colors, edgecolor="black", alpha=0.85)
for i, v in enumerate(target_counts.values):
    axes[0].text(i, v + 150, f"{v:,} ({v/len(df):.2%})", ha="center", fontweight="bold")
axes[0].set_title("Overall Machine Failure Distribution (Binary Target)", fontsize=12, fontweight="bold")
axes[0].set_ylabel("Record Count")
axes[0].set_ylim(0, 11000)

# Failure modes breakdown
failure_modes = {"Tool Wear (TWF)": df["twf"].sum(), "Heat Dissipation (HDF)": df["hdf"].sum(),
                 "Power (PWF)": df["pwf"].sum(), "Overstrain (OSF)": df["osf"].sum(),
                 "Random (RNF)": df["rnf"].sum()}
axes[1].barh(list(failure_modes.keys()), list(failure_modes.values()), color="#e67e22", edgecolor="black", alpha=0.85)
for i, v in enumerate(failure_modes.values()):
    axes[1].text(v + 2, i, f"{v}", va="center", fontweight="bold")
axes[1].set_title("Breakdown by Specific Failure Modes", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Count of Occurrences")
plt.tight_layout()
plt.show()

#### Key Evaluation Takeaways:
- **Accuracy is a deceptive metric**: A naive dummy model predicting all zeros achieves $96.61\%$ accuracy while catching $0\%$ of catastrophic failures.
- **Required Primary Metrics**: **Precision-Recall Area Under Curve (PR-AUC)**, **F1-Score (Macro & Minority)**, and **Recall@Top-K**.

### 4. Categorical Feature Analysis (`Type`)
The dataset includes machine quality variants: `L` (Low, 50%), `M` (Medium, 30%), and `H` (High, 20%).

In [ ]:
type_counts = df["type"].value_counts()
type_failure_rates = df.groupby("type")["machine_failure"].mean() * 100

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].pie(type_counts.values, labels=[f"{k} ({v:,})" for k, v in type_counts.items()],
            autopct="%1.1f%%", colors=["#3498db", "#2ecc71", "#9b59b6"], startangle=140)
axes[0].set_title("Machine Quality Variant Distribution", fontsize=12, fontweight="bold")

axes[1].bar(type_failure_rates.index, type_failure_rates.values, color=["#e74c3c", "#f39c12", "#1abc9c"], edgecolor="black", alpha=0.85)
for i, v in enumerate(type_failure_rates.values):
    axes[1].text(i, v + 0.1, f"{v:.2f}%", ha="center", fontweight="bold")
axes[1].set_title("Failure Rate by Machine Quality Variant", fontsize=12, fontweight="bold")
axes[1].set_ylabel("Failure Rate (%)")
axes[1].set_ylim(0, 5)
plt.tight_layout()
plt.show()

**Observation**: Type `L` (Low quality) exhibits the highest failure rate ($3.92\%$), while Type `H` (High quality) is the most resilient ($2.00\%$).

### 5. Numerical Feature Distributions & Outliers
We examine the continuous sensor telemetry: Air Temperature, Process Temperature, Rotational Speed, Torque, and Tool Wear.

In [ ]:
num_cols = ["air_temperature_k", "process_temperature_k", "rotational_speed_rpm", "torque_nm", "tool_wear_min"]

fig, axes = plt.subplots(len(num_cols), 2, figsize=(14, 18))
for i, col in enumerate(num_cols):
    # Distribution (Histogram + KDE)
    sns.histplot(df[col], kde=True, ax=axes[i, 0], color="#2980b9", bins=30)
    axes[i, 0].set_title(f"{col} Distribution (Skew: {df[col].skew():.2f})", fontweight="bold")
    axes[i, 0].set_ylabel("Density")
    
    # Boxplot segmented by target
    sns.boxplot(x="machine_failure", y=col, data=df, ax=axes[i, 1], palette=["#3498db", "#e74c3c"])
    axes[i, 1].set_title(f"{col} by Machine Failure Status", fontweight="bold")
    axes[i, 1].set_xticklabels(["Normal (0)", "Failure (1)"])

plt.tight_layout()
plt.show()

#### Distribution Insights:
1. **Air & Process Temperatures**: Symmetrically distributed ($~300\text{ K}$ and $~310\text{ K}$) with near-zero skewness.
2. **Rotational Speed**: Positively skewed ($+1.99$), containing prominent upper outliers corresponding to low torque operations.
3. **Torque**: Symmetric bell curve ($~40\text{ Nm}$), but failures occur heavily at extreme high values ($>60\text{ Nm}$) and extreme low values ($<15\text{ Nm}$).
4. **Tool Wear**: Uniformly distributed ($0$ to $250\text{ min}$); failures concentrate heavily when wear exceeds $200\text{ min}$.

### 6. Correlation & Multi-Collinearity Analysis
We analyze linear (Pearson) and non-linear monotonic (Spearman) relationships among features.

In [ ]:
derived_df = compute_domain_features(df)
corr_features = num_cols + ["power_w", "temp_difference_k", "overstrain_index", "machine_failure"]
corr_matrix = derived_df[corr_features].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0, vmin=-1, vmax=1, square=True, linewidths=0.5)
plt.title("Pearson Correlation Matrix (with Physics-Derived Features)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

**Key Physical Correlation Found**:
- `rotational_speed_rpm` and `torque_nm` exhibit a strong negative correlation ($\rho = -0.88$). This directly reflects mechanical cutting power: $P = \tau \cdot \omega$.
- `process_temperature_k` and `air_temperature_k` are strongly coupled ($\rho = 0.88$).

### 7. Multi-Variate Sensor Failure Boundaries
Let's visualize the exact 2D operational failure boundaries across the primary physical failure regimes.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# 1. Torque vs Rotational Speed (Power Failure Boundary)
sns.scatterplot(x="rotational_speed_rpm", y="torque_nm", hue="machine_failure",
                data=df, palette=["#3498db", "#e74c3c"], alpha=0.7, ax=axes[0], s=30)
axes[0].set_title("Torque vs. Rotational Speed (PWF Regime)", fontweight="bold")
axes[0].set_xlabel("Rotational Speed [rpm]")
axes[0].set_ylabel("Torque [Nm]")

# 2. Tool Wear vs Torque (Overstrain Failure Boundary)
sns.scatterplot(x="tool_wear_min", y="torque_nm", hue="machine_failure",
                data=df, palette=["#3498db", "#e74c3c"], alpha=0.7, ax=axes[1], s=30)
axes[1].set_title("Torque vs. Tool Wear (OSF Regime)", fontweight="bold")
axes[1].set_xlabel("Tool Wear [min]")
axes[1].set_ylabel("Torque [Nm]")

plt.tight_layout()
plt.show()

### 8. Observations vs. Assumptions Summary

| Area | Verifiable Observation (Data Evidence) | Modeling Assumption (Hypothesis) |
| :--- | :--- | :--- |
| **Class Balance** | Failures account for 3.39% (339 / 10,000). | Class-weighted loss or focal loss will substantially improve minority recall over naive cross-entropy. |
| **Sensor Coupling** | Torque and Speed are inversely correlated ($-0.88$). | Computing mechanical power $P = \tau \cdot \omega$ provides an explicit decision boundary for Power Failures (PWF). |
| **Thermal Regime** | Failures cluster when process-to-air temperature difference drops below $8.6\text{ K}$ at low RPM. | Engineering $\Delta T = T_{\text{process}} - T_{\text{air}}$ will strongly capture Heat Dissipation Failures (HDF). |
| **Tool Wear Strain** | High torque combined with tool wear $>200\text{ min}$ accounts for overstrain failures. | Interaction term $\text{Torque} \times \text{Tool Wear}$ is a primary predictive signal for OSF and TWF. |
| **Quality Type** | Type L has $3.92\%$ failure rate vs $2.00\%$ in Type H. | One-hot or ordinal encoding of Type will help tree models segment baselines per quality grade. |